# Deep Learning — Complete Cheatsheet (Mod 1–6, All-in-One)

This single notebook merges **all 6 Phitron DL modules** into one runnable, top-to-bottom narrative,
plus a **bonus "Beyond the Modules" section** covering standard DL fundamentals the modules didn't
reach yet (backprop walkthrough, optimizers, regularization, weight init).

**Modules covered:**
1. ML vs DL, biological neuron -> artificial neuron, why non-linearity
2. The Perceptron (structure, learning rule, AND/OR/XOR)
3. Perceptron advantages & disadvantages
4. Decision boundaries as line coefficients + error-driven updates
5. Gradient Descent (linear regression from scratch)
6. Loss functions (Hinge / Perceptron loss, Binary Cross-Entropy)
7. Activation functions (the full zoo: sigmoid -> softsign)
8. The Multi-Layer Perceptron (MLP) — notation, forward propagation, capstone XOR + spiral demo
9. Quick-reference summary
10. **Bonus:** backpropagation, vanishing/exploding gradients, weight init, optimizers, regularization

**Dependencies:** only `numpy`, `matplotlib`, and `pandas` — no sklearn/tensorflow required, so every
cell runs anywhere.

```
pip install numpy pandas matplotlib
```

Run cells top to bottom — later sections reuse helper functions defined earlier.

---
## 0. Setup — imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("fivethirtyeight")
np.random.seed(42)

---
# 1. ML vs DL, the Biological Neuron, and Why Non-Linearity (Mod 1)

## 1.1 Why Deep Learning?
Classical ML needs **manual feature engineering** — a human decides which columns/transformations
matter (e.g. for price prediction: car name, mileage, engine size...). This works fine when:

- the **dataset is small**, and performance plateaus quickly with more data,
- the **pattern is simple / near-linear**,
- domain knowledge is available to hand-craft features.

Deep Learning instead **learns its own feature representations** directly from raw data
(pixels, text, audio), and its performance keeps improving as more data and compute are thrown at it.
It shines when:

1. Feature engineering is hard (images, free text, audio, video).
2. The dataset is large.
3. The underlying pattern is **complex / non-linear**.

**Common DL application areas:** recommendation systems (Spotify, Netflix), computer vision,
NLP (Transformers), sequence-to-sequence problems (translation, time series, `x(t) -> x(t+1)`).

## 1.2 Biological Neuron -> Artificial Neuron
McCulloch & Pitts (1943) showed that a simple mathematical model of a neuron can reproduce basic
logical operations (AND, OR, NOT). Rosenblatt (1957) extended this into the **Perceptron**.

| Biological neuron | Artificial neuron |
|---|---|
| Dendrites (receive signals) | Inputs $x_1, x_2, \dots, x_n$ |
| Synapse strength | Weights $w_1, w_2, \dots, w_n$ |
| Soma (sums signals) | Weighted sum $z = \sum_i w_i x_i + b$ |
| Fires an action potential past a threshold | Activation function $f(z)$ |
| Axon (output) | Output $y$ |

A biological neuron only fires (sends a signal down the axon) once its accumulated input crosses a
threshold — this is exactly the **step activation function** used in the first artificial neurons.

## 1.3 Why non-linearity matters
A network built only from linear layers ($z = w^Tx + b$ stacked on $z = w^Tx+b$...) collapses back
into **one single linear function**, no matter how many layers you stack — because a linear
combination of linear functions is still linear. Real-world data is rarely linearly separable
(two intertwined spirals, concentric circles, XOR, etc.), so we need a **non-linear activation
function** between layers to let the network bend its decision boundary.

Below we generate the classic **two-spiral dataset** (non-linearly separable) and show that a purely
linear classifier (logistic regression, i.e. one perceptron-like unit with a sigmoid) cannot solve
it — motivating everything that follows (Sections 2–8 build up to a hand-written MLP that *does*
solve it, at the end of Section 8).

In [ ]:
def generate_spiral(n, noise=0.2):
    # r = distance from center, theta = angle -> classic parametric spiral
    theta = np.sqrt(np.random.rand(n)) * 2 * np.pi
    r = 2 * theta
    x1 = r * np.cos(theta) + noise * np.random.randn(n)
    x2 = r * np.sin(theta) + noise * np.random.randn(n)
    return np.c_[x1, x2]

X0 = generate_spiral(300)
X1 = -generate_spiral(300)

X_spiral = np.vstack([X0, X1])
y_spiral = np.array([0]*300 + [1]*300)

plt.figure(figsize=(5, 5))
plt.scatter(X_spiral[y_spiral == 0][:, 0], X_spiral[y_spiral == 0][:, 1], color="red", s=10, label="Class 0")
plt.scatter(X_spiral[y_spiral == 1][:, 0], X_spiral[y_spiral == 1][:, 1], color="blue", s=10, label="Class 1")
plt.title("Two-Spiral Dataset (non-linearly separable)")
plt.legend()
plt.show()

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logistic_regression_gd(X, y, lr=0.05, epochs=500):
    '''A single linear unit + sigmoid, trained with gradient descent on BCE loss.'''
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        y_hat = sigmoid(z)
        error = y_hat - y
        w -= lr * (X.T @ error) / n
        b -= lr * error.mean()
    return w, b

w_lin, b_lin = logistic_regression_gd(X_spiral, y_spiral)
preds = (sigmoid(X_spiral @ w_lin + b_lin) >= 0.5).astype(int)
print("Linear-classifier accuracy on the spiral dataset:", (preds == y_spiral).mean())
print("-> a single linear unit cannot separate the spirals. We revisit this with a full MLP at the end of Section 8.")

---
# 2. The Perceptron (Mod 1 & Mod 2)

## 2.1 Structure
$$z = w_0 x_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n, \qquad x_0 = 1 \text{ (bias trick)}$$

The perceptron applies a **step (Heaviside) activation** on top of the linear sum:

$$\phi(z) = \begin{cases} 1 & z \geq 0 \\ 0 & z < 0 \end{cases}$$

(An alternative convention uses $sgn(z) \in \{-1, 0, 1\}$.)

## 2.2 The Perceptron Learning Rule
$$w_{new} = w_{old} + \eta\,(y - \hat{y})\,x_i \qquad b_{new} = b_{old} + \eta\,(y-\hat{y})$$

- $\eta$ = learning rate $\in (0, 1]$ (a common example value used in the notes: $\eta = 0.5$)
- $y$ = actual/target label, $\hat{y}$ = predicted label, $x_i$ = the $i^{th}$ input
- The term $(y - \hat{y})$ is the **error**: it is $0$ when the prediction is already correct
  (no update), $+1$ or $-1$ otherwise.

**Rosenblatt's Perceptron Convergence Theorem:** if the data is **linearly separable**, this rule is
guaranteed to converge to a set of weights that perfectly classifies it. If the data is *not*
linearly separable (e.g. XOR), it will **never converge**.

### Worked numeric example (from the course notes)
$w_1 = 1.2,\ w_2 = 0.6,\ \eta = 0.5,\ threshold = 1$. Input $x_1=1, x_2=0$:

$$w_1 x_1 + w_2 x_2 = 1.2 \times 1 + 0.6 \times 0 = 1.2 > 1 \Rightarrow \hat{y} = 1$$

If the true label is $y = 0$, error $= y - \hat{y} = -1$, so:

$$w_{1,new} = 1.2 + 0.5 \times (-1) \times 1 = 0.7 \qquad w_{2,new} = 0.6 + 0.5\times(-1)\times 0 = 0.6$$

(only $w_1$ changes, because $x_2 = 0$ contributes nothing to its own update.)

## 2.3 AND gate truth table (linearly separable)

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

In [ ]:
def step(x):
    return np.where(x >= 0, 1, 0)

x_axis = np.linspace(-2, 2, 200)
plt.plot(x_axis, step(x_axis))
plt.title("Step / Heaviside activation")
plt.xlabel("z"); plt.ylabel("step(z)")
plt.axhline(0, color="k", lw=1); plt.axvline(0, color="k", lw=1)
plt.show()

In [ ]:
class Perceptron:
    '''From-scratch perceptron: bias folded in as an extra "-1" feature column.'''
    def __init__(self, eta, epochs):
        self.weights = np.random.randn(3) * 1e-4
        self.eta = eta
        self.epochs = epochs

    def activation(self, inputs, weights):
        z = np.dot(inputs, weights)
        return np.where(z > 0, 1, 0)

    def fit(self, X, y):
        X_with_bias = np.c_[X, -np.ones((len(X), 1))]
        for epoch in range(self.epochs):
            y_hat = self.activation(X_with_bias, self.weights)
            error = y - y_hat
            self.weights = self.weights + self.eta * np.dot(X_with_bias.T, error)
        return self

    def predict(self, X):
        X_with_bias = np.c_[X, -np.ones((len(X), 1))]
        return self.activation(X_with_bias, self.weights)

AND = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1], "y": [0, 0, 0, 1]})
X_and, y_and = AND[["x1", "x2"]], AND["y"]

and_model = Perceptron(eta=0.5, epochs=10).fit(X_and, y_and)
print("AND predictions:", and_model.predict(X_and))
print("AND learned weights [w1, w2, bias]:", and_model.weights)

In [ ]:
OR = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1], "y": [0, 1, 1, 1]})
X_or, y_or = OR[["x1", "x2"]], OR["y"]

or_model = Perceptron(eta=0.5, epochs=10).fit(X_or, y_or)
print("OR predictions:", or_model.predict(X_or))
print("OR learned weights [w1, w2, bias]:", or_model.weights)

In [ ]:
XOR = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 1, 0], "y": [0, 1, 1, 0]})
X_xor, y_xor = XOR[["x1", "x2"]], XOR["y"]

xor_model = Perceptron(eta=0.5, epochs=50).fit(X_xor, y_xor)
print("XOR predictions:", xor_model.predict(X_xor))
print("XOR true labels:  ", y_xor.values)
print("-> the single perceptron CANNOT learn XOR: it's not linearly separable. See Section 8 for the MLP fix.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (df, title) in zip(axes, [(AND, "AND (separable)"), (OR, "OR (separable)"), (XOR, "XOR (NOT separable)")]):
    ax.scatter(df["x1"], df["x2"], c=df["y"], s=200, cmap="winter", edgecolor="k")
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)

# AND / OR: one straight line separates the classes
x_line = np.linspace(-0.5, 1.5, 50)
axes[0].plot(x_line, 1.5 - 1 * x_line, "r--")
axes[1].plot(x_line, 0.5 - 1 * x_line, "r--")
plt.tight_layout()
plt.show()

---
# 3. Advantages & Disadvantages of the (Single-Layer) Perceptron (Mod 5)

| Advantages | Disadvantages |
|---|---|
| Simple and easy to understand | Can only solve **linearly separable** problems |
| Fast training | **Cannot solve XOR** |
| Computationally efficient | Uses a step function (**not differentiable**) — blocks gradient-based training |
| Works well for simple binary classification | Does not provide a probability output |
| Easy mathematical formulation | Limited expressive power (**no hidden layer**) |

The non-differentiability of the step function is the key blocker: it means we **cannot use
calculus-based optimization (gradient descent)** to train it — only the discrete perceptron rule.
Section 5 introduces gradient descent, and Section 7 introduces *smooth, differentiable* activation
functions that make it possible to train deeper networks.

---
# 4. Decision Boundaries as Line Coefficients (Mod 3)

The perceptron's decision boundary $w_1x_1 + w_2x_2 + b = 0$ is just the general form of a straight
line, $Ax + By + C = 0$. The same **error-driven update rule** from Section 2 generalizes directly to
each of the line's coefficients:

$$Coeff_{new} = Coeff_{old} + \eta \cdot x_i \cdot error \qquad (error = y - \hat{y})$$

**Error measurement for a line/perceptron boundary = number of misclassified points.** Every
training pass counts how many points fall on the wrong side of the current line, and the
coefficients $A, B, C$ shift a little (scaled by $\eta$) to reduce that count, exactly like nudging
$w_1, w_2, b$ in Section 2.

This is the bridge to Section 5: instead of nudging coefficients by a fixed step per misclassified
point (perceptron rule), gradient descent nudges them by the **calculus-derived slope of a smooth
loss function** — which works even when classes overlap or aren't perfectly linearly separable.

In [ ]:
# Visualize a 2D decision boundary Ax + By + C = 0 with points on both sides
np.random.seed(1)
blue = np.random.randn(40, 2) + np.array([-2, -2])
green = np.random.randn(40, 2) + np.array([2, 2])

A, B, C = 1, 1, 0  # decision line: x + y = 0

plt.figure(figsize=(5, 5))
plt.scatter(blue[:, 0], blue[:, 1], color="blue", label="Class 0")
plt.scatter(green[:, 0], green[:, 1], color="green", label="Class 1")
xs = np.linspace(-5, 5, 50)
plt.plot(xs, -(A * xs + C) / B, "r--", label=f"{A}x + {B}y + {C} = 0")
plt.legend(); plt.title("Decision boundary as line coefficients")
plt.show()

---
# 5. Gradient Descent (Mod 4)

## 5.1 From counting errors to minimizing a loss
For a simple linear-regression example (study hours $x$ predicting attendance $y$, fit line
$\hat{y} = mx + b$), define the **sum-of-squared-errors loss**:

$$L(m, b) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \sum_{i=1}^{n} (y_i - m x_i - b)^2$$

Plotting $L$ against $b$ (holding $m$ fixed) traces a **parabola** with a single **global minimum** —
gradient descent's whole job is to walk down that bowl (the classic "person walking down a foggy
mountain toward the valley" analogy) by repeatedly stepping opposite the slope.

## 5.2 The update rule
$$b_{new} = b_{old} - \eta \cdot slope = b_{old} - \eta \frac{\partial L}{\partial b}$$
$$m_{new} = m_{old} - \eta \cdot \frac{\partial L}{\partial m}$$

- A **positive slope** means the minimum is to the *left* -> subtract to move left.
- A **negative slope** means the minimum is to the *right* -> subtracting a negative *adds*, moving right.
- This is why the update is always $-\eta \times \text{slope}$, regardless of the slope's sign.

## 5.3 Deriving the partial derivatives
$$\frac{\partial L}{\partial b} = -2\sum_{i=1}^n (y_i - m x_i - b)$$
$$\frac{\partial L}{\partial m} = -2\sum_{i=1}^n (y_i - m x_i - b)\,x_i$$

Substituting back: $m_{new} = m_{old} - \eta \times slope_m$, $b_{new} = b_{old} - \eta \times slope_b$.

## 5.4 Perceptron rule vs. Gradient Descent

| Feature | Perceptron Learning Rule | Gradient Descent |
|---|---|---|
| Data requirement | Requires data to be **linearly separable** to converge | Can handle overlapping / non-linearly separable data |
| Update method | Discrete: $w \leftarrow w + \eta(y-\hat y)x$ | Smooth: $w \leftarrow w - \eta\,\partial L/\partial w$ |
| Convergence guarantee | Only if linearly separable | Converges to a minimum of the loss even if perfect separation isn't possible |
| Step control | Fixed learning rate can overshoot | Learning rate can be tuned; momentum/scheduling can help |
| Error measurement | Implicit — only right/wrong count | Explicit — minimizes a continuous loss (MSE, cross-entropy, ...) |
| Flexibility | Single-layer perceptrons only | Works for single-layer **and multi-layer (MLP)** networks |

## 5.5 From-scratch gradient descent regressor

In [ ]:
class GDRegressor:
    def __init__(self, learning_rate, epochs):
        self.m = 1.0
        self.b = 0.0
        self.lr = learning_rate
        self.epochs = epochs
        self.history = []

    def fit(self, X, y):
        X = X.ravel()
        for _ in range(self.epochs):
            loss_slope_b = -2 * np.sum(y - self.m * X - self.b)
            loss_slope_m = -2 * np.sum((y - self.m * X - self.b) * X)
            self.m -= self.lr * loss_slope_m
            self.b -= self.lr * loss_slope_b
            loss = np.sum((y - self.m * X - self.b) ** 2)
            self.history.append((self.m, self.b, loss))
        return self

    def predict(self, X):
        return self.m * X.ravel() + self.b


# synthetic linear data: y = 30x - 2 + noise
np.random.seed(13)
X_reg = np.random.rand(100, 1) * 4 - 2
true_m, true_b = 30, -2
y_reg = true_m * X_reg.ravel() + true_b + np.random.randn(100) * 20

gd = GDRegressor(learning_rate=0.001, epochs=50)
gd.fit(X_reg, y_reg)
print(f"True   m = {true_m}, b = {true_b}")
print(f"Learned m = {gd.m:.3f}, b = {gd.b:.3f}")

In [ ]:
hist = np.array(gd.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(hist[:, 2])
axes[0].set_title("Loss vs. epoch (descending the bowl)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("L(m, b)")

axes[1].scatter(X_reg, y_reg, alpha=0.5, label="data")
xs = np.linspace(X_reg.min(), X_reg.max(), 50)
axes[1].plot(xs, gd.m * xs + gd.b, "r-", label="fitted line")
axes[1].set_title("Fitted regression line")
axes[1].legend()
plt.tight_layout()
plt.show()

---
# 6. Loss Functions (Mod 4 & Mod 5)

## 6.1 Mean Squared Error (MSE) — regression
Already used above: $L = \frac{1}{n}\sum (y_i - \hat y_i)^2$. Penalizes large errors quadratically.

## 6.2 Hinge / Perceptron Loss — classification with labels $\{+1, -1\}$
$$L = \frac{1}{n}\sum_{i=1}^n \max\big(0,\ -y_i f(x_i)\big), \qquad f(x_i) = w_1 x_{i1} + w_2 x_{i2} + b$$

- If the point is **correctly classified** ($y_i$ and $f(x_i)$ share the same sign), $-y_if(x_i) < 0$
  so the loss contribution is $\max(0, \text{negative}) = 0$.
- If **misclassified** (opposite signs), $-y_if(x_i) > 0$ and the loss grows with how wrong it is.

Sub-gradient (used to derive the perceptron weight-update rule from Section 2 via gradient descent):
$$\frac{\partial L}{\partial w_1} = \begin{cases} 0 & y_i f(x_i) \geq 0 \\ -y_i x_{i1} & y_i f(x_i) < 0\end{cases}$$
(analogous formulas hold for $w_2$ and $b$). This is exactly how the discrete perceptron rule in
Section 2 can be re-derived as a special case of gradient descent on a non-smooth loss.

## 6.3 Binary Cross-Entropy (BCE) — probabilistic classification
Derived from **Maximum Likelihood**: given predicted probabilities, prefer the model whose predicted
probability of the *true* class is highest. Using $\log(a \cdot b) = \log a + \log b$ turns a product
of probabilities into a sum of logs (numerically stable, and easy to differentiate), and flipping the
sign turns "maximize likelihood" into "minimize negative log-likelihood":

$$L(y, \hat y) = -\Big[y \log(\hat y) + (1-y)\log(1-\hat y)\Big]$$

- If $y=1$: loss $= -\log(\hat y)$ — the loss is small when $\hat y$ is close to 1 (confident and correct).
- If $y=0$: loss $= -\log(1-\hat y)$ — small when $\hat y$ is close to 0.

Worked example: $y=1$, predicted $\hat y = 0.7 \Rightarrow loss = -\log(0.7) \approx 0.357$.

In [ ]:
def binary_cross_entropy(y, y_hat, eps=1e-12):
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

print("y=1, y_hat=0.7 ->", binary_cross_entropy(1, 0.7))
print("y=1, y_hat=0.99 ->", binary_cross_entropy(1, 0.99))
print("y=0, y_hat=0.7 ->", binary_cross_entropy(0, 0.7))

In [ ]:
p = np.linspace(0.001, 0.999, 200)
plt.plot(p, binary_cross_entropy(1, p), label="y = 1: $-\\log(\\hat y)$")
plt.plot(p, binary_cross_entropy(0, p), label="y = 0: $-\\log(1-\\hat y)$")
plt.xlabel(r"predicted probability $\hat y$"); plt.ylabel("BCE loss")
plt.title("Binary Cross-Entropy vs. predicted probability")
plt.legend(); plt.show()

---
# 7. Activation Functions — The Full Zoo (Mod 5)

| Function | Formula | Range | Derivative | Typical use |
|---|---|---|---|---|
| Step | $1$ if $z\geq0$ else $0$ | $\{0,1\}$ | undefined at 0, else 0 | classic perceptron only |
| Sigmoid | $\sigma(x)=\frac{1}{1+e^{-x}}$ | $(0,1)$ | $\sigma(x)(1-\sigma(x))$ | binary output layer |
| Tanh | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $(-1,1)$ | $1-\tanh^2(x)$ | zero-centered hidden layers |
| ReLU | $\max(0,x)$ | $[0,\infty)$ | $1$ if $x>0$ else $0$ | default hidden-layer choice |
| Leaky ReLU | $x$ if $x\geq0$ else $\alpha x$ | $(-\infty,\infty)$ | $1$ or $\alpha$ | fixes "dead ReLU" |
| ELU | $x$ if $x\geq0$ else $\alpha(e^x-1)$ | $(-\alpha,\infty)$ | smooth | zero-centered, no dead units |
| Softmax | $\frac{e^{x_j}}{\sum_k e^{x_k}}$ | $(0,1)$, sums to 1 | Jacobian (see Sec. 10) | multi-class output layer |
| Swish | $x\cdot\sigma(x)$ | $\approx(-0.28,\infty)$ | smooth | modern deep nets |
| Softplus | $\log(1+e^x)$ | $(0,\infty)$ | $\sigma(x)$ | smooth ReLU approximation |
| Softsign | $\frac{x}{1+|x|}$ | $(-1,1)$ | smooth | cheaper tanh alternative |

## 7.1 Sigmoid
$$\sigma(x) = \frac{1}{1+e^{-x}}, \qquad x \in (-\infty,\infty),\ \sigma(x)\in(0,1)$$
Used with **threshold 0.5** for binary classification. Its derivative has a famously clean form,
derived below by the quotient rule:
$$\sigma'(x) = \sigma(x)\big(1-\sigma(x)\big)$$

**Advantages:** smooth, easy to derive, output interpretable as a firing-rate/probability-like signal.
**Disadvantages:**
1. **Gradient saturation** — far from 0, the gradient is nearly zero, so backprop through many
   sigmoid layers makes early-layer weights barely update.
2. Output is **not zero-centered**, which slows weight updates.
3. Involves an expensive exponential — slower to compute than ReLU.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

x = np.linspace(-10, 10, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, sigmoid(x)); axes[0].set_title(r"$\sigma(x) = 1/(1+e^{-x})$")
axes[1].plot(x, sigmoid_derivative(x)); axes[1].set_title(r"$\sigma'(x) = \sigma(x)(1-\sigma(x))$")
for ax in axes:
    ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)
plt.tight_layout(); plt.show()

## 7.2 Tanh
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}, \qquad x\in(-\infty,\infty),\ \tanh(x)\in(-1,1)$$
Derivative (via the quotient rule on $u=e^x-e^{-x}$, $v=e^x+e^{-x}$):
$$\tanh'(x) = 1 - \tanh^2(x)$$
Zero-centered (unlike sigmoid), so it's generally preferred over sigmoid for hidden layers — though
it still saturates for large $|x|$.

In [ ]:
def tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def tanh_derivative(x):
    return 1 - tanh(x) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, tanh(x)); axes[0].set_title("tanh(x)")
axes[1].plot(x, tanh_derivative(x)); axes[1].set_title(r"$1-\tanh^2(x)$")
for ax in axes:
    ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)
plt.tight_layout(); plt.show()

## 7.3 ReLU (Rectified Linear Unit)
$$f(x) = \max(0, x) = \begin{cases} 0 & x \le 0 \\ x & x > 0\end{cases}$$
**Advantages:** no gradient saturation for $x>0$; much cheaper to compute than sigmoid/tanh (no
exponential); the default choice for hidden layers today.
**Disadvantages:** "**dying ReLU**" — a unit that always outputs $\le 0$ has zero gradient forever and
stops learning; output is not zero-centered.

## 7.4 Leaky ReLU
$$f(x) = \begin{cases} x & x \ge 0 \\ \alpha x & x < 0 \end{cases}, \quad \alpha \approx 0.01\text{–}0.3$$
Fixes dying ReLU by giving negative inputs a small, non-zero gradient $\alpha$.
When $\alpha$ is itself a *learnable* parameter, this becomes **PReLU** (Parametric ReLU).

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def leaky_relu(x, alpha=0.1):
    return np.where(x >= 0, x, alpha * x)

def leaky_relu_derivative(x, alpha=0.1):
    return np.where(x >= 0, 1.0, alpha)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0,0].plot(x, relu(x)); axes[0,0].set_title("ReLU")
axes[0,1].plot(x, relu_derivative(x)); axes[0,1].set_title("ReLU'")
axes[1,0].plot(x, leaky_relu(x)); axes[1,0].set_title("Leaky ReLU (alpha=0.1)")
axes[1,1].plot(x, leaky_relu_derivative(x)); axes[1,1].set_title("Leaky ReLU'")
for ax in axes.ravel():
    ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)
plt.tight_layout(); plt.show()

## 7.5 ELU (Exponential Linear Unit)
$$f(x) = \begin{cases} x & x\ge0 \\ \alpha(e^x - 1) & x<0 \end{cases}$$
Keeps all of ReLU's advantages, has no dead-unit problem, and its output mean is closer to zero — at
the cost of a slightly more expensive exponential for negative inputs.

In [ ]:
def elu(x, alpha=1.0):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

plt.plot(x, elu(x)); plt.title("ELU"); plt.axhline(0, color="k", lw=1); plt.axvline(0, color="k", lw=1)
plt.show()

## 7.6 Softmax — multi-class output layer
$$S(x_j) = \frac{e^{x_j}}{\sum_{k=1}^K e^{x_k}}, \qquad j=1,\dots,K$$
Squashes a $K$-length real vector ("logits") into a probability distribution that sums to 1. Unlike
plain `max`, it's a *soft* max — smaller logits still get some non-zero probability instead of being
discarded outright.

**Worked numeric example** (3-class: cat / dog / horse), logits $z = [2, 1, 0]$:
$$e^2=7.39,\quad e^1=2.72,\quad e^0=1, \qquad \text{sum}=11.11$$
$$P(\text{cat})=\frac{7.39}{11.11}\approx0.665,\quad P(\text{dog})=\frac{2.72}{11.11}\approx0.245,\quad P(\text{horse})=\frac{1}{11.11}\approx0.090$$

In [ ]:
def softmax(z):
    z = z - np.max(z)  # numerical stability, doesn't change the result
    e = np.exp(z)
    return e / e.sum()

z_logits = np.array([2.0, 1.0, 0.0])
probs = softmax(z_logits)
for label, p in zip(["cat", "dog", "horse"], probs):
    print(f"{label:6s}: {p:.3f}")
print("sum:", probs.sum())

## 7.7 A few more you'll meet in the wild (same family, quick reference)
- **Swish:** $f(x) = x \cdot \sigma(\beta x)$ — smooth, non-monotonic, used in many modern architectures.
- **Softplus:** $f(x) = \log(1+e^x)$ — a smooth, differentiable approximation of ReLU (its derivative is sigmoid).
- **Softsign:** $f(x) = \frac{x}{1+|x|}$ — cheaper, saturating alternative to tanh.
- **Maxout:** $f(x) = \max(w_1^Tx+b_1,\ w_2^Tx+b_2)$ — generalizes ReLU/Leaky ReLU but doubles parameters per neuron.

In [ ]:
def swish(x, beta=1.0):
    return x * sigmoid(beta * x)

def softplus(x):
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)  # numerically stable log(1+e^x)

def softsign(x):
    return x / (1 + np.abs(x))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(x, swish(x)); axes[0].set_title("Swish")
axes[1].plot(x, softplus(x)); axes[1].set_title("Softplus")
axes[2].plot(x, softsign(x)); axes[2].set_title("Softsign")
for ax in axes:
    ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)
plt.tight_layout(); plt.show()

---
# 8. The Multi-Layer Perceptron — Main Idea & Notation (Mod 6)

## 8.1 The main idea: why stacking perceptrons works
A single perceptron can only draw **one straight decision boundary**. But if you take two
perceptrons that each draw a *different* line, and feed their outputs into a **third** perceptron
that combines them, the combination can carve out a **non-linear region** — e.g. a wedge, a corner,
or (with more hidden units) arbitrarily complex shapes. This is exactly how a hidden layer lets a
network solve XOR: two lines, one hidden layer, one combining output unit.

## 8.2 Architecture
```
x1 -\
x2 -+-> hidden layer 1 -> hidden layer 2 -> output
x3 -/
```
Each arrow is a full set of weighted connections (a small "dense"/"fully-connected" layer).

## 8.3 Notation
- **Bias:** $b_{ij}$ where $i$ = layer number, $j$ = node index within that layer.
- **Weight:** $W^{k}_{ij}$ where $k$ = layer number, $i$ = source-node index (previous layer),
  $j$ = destination-node index (this layer).
- **Layer output / activation:** $a^{[l]} = \sigma\big(a^{[l-1]} \cdot W^{[l]} + b^{[l]}\big)$
  — this recursive formula is **forward propagation**: apply a linear transform, then an activation,
  layer after layer, from input to output.

### Worked example — counting trainable parameters
For a network with **4 inputs -> 3 hidden(layer 1) -> 2 hidden(layer 2) -> 1 output**:

$$W: \underbrace{4\times3}_{12} + \underbrace{3\times2}_{6} + \underbrace{2\times1}_{2} = 20 \qquad
b: \underbrace{3}_{layer\,1} + \underbrace{2}_{layer\,2} + \underbrace{1}_{output} = 6$$

$$\text{Total trainable parameters} = 20 + 6 = 26$$

### Worked example — forward propagation in matrix form
Layer 1 (4 inputs -> 3 hidden units): with input vector $x_i \in \mathbb{R}^4$ and weight matrix
$W^{[1]} \in \mathbb{R}^{4\times3}$,
$$a^{[1]} = \sigma\big({W^{[1]}}^T x_i + b^{[1]}\big) \in \mathbb{R}^3$$
Layer 2 (3 -> 2 hidden units): $a^{[2]} = \sigma\big({W^{[2]}}^T a^{[1]} + b^{[2]}\big) \in \mathbb{R}^2$

Output layer (2 -> 1): $\hat y = a^{[3]} = \sigma\big({W^{[3]}}^T a^{[2]} + b^{[3]}\big) \in \mathbb{R}$

We reproduce this exact shape below with numpy.

In [ ]:
np.random.seed(0)

x_i = np.random.randn(4)          # 4 inputs
W1, b1 = np.random.randn(4, 3) * 0.5, np.random.randn(3) * 0.1   # layer 1: 4 -> 3
W2, b2 = np.random.randn(3, 2) * 0.5, np.random.randn(2) * 0.1   # layer 2: 3 -> 2
W3, b3 = np.random.randn(2, 1) * 0.5, np.random.randn(1) * 0.1   # output:  2 -> 1

a0 = x_i
a1 = sigmoid(a0 @ W1 + b1)
a2 = sigmoid(a1 @ W2 + b2)
a3 = sigmoid(a2 @ W3 + b3)   # = y_hat

print("a0 (input)      :", a0)
print("a1 (hidden 1, 3):", a1)
print("a2 (hidden 2, 2):", a2)
print("a3 = y_hat      :", a3)

n_params = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print("\nTrainable parameters:", n_params, "(matches 20 weights + 6 biases = 26 for a 4-3-2-1 network)")

## 8.4 Capstone 1 — a from-scratch MLP solving XOR
Section 2 showed the single perceptron **cannot** learn XOR. Here's a minimal 2-layer MLP
(2 inputs -> 2 hidden sigmoid units -> 1 output sigmoid unit) trained with plain gradient descent /
backpropagation (chain rule — derived in detail in Section 10.1) that **can**.

In [ ]:
class SimpleMLP:
    def __init__(self, n_in, n_hidden, n_out, lr=0.5, seed=1):
        rng = np.random.default_rng(seed)
        self.W1 = rng.standard_normal((n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.standard_normal((n_hidden, n_out))
        self.b2 = np.zeros(n_out)
        self.lr = lr

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)     # y_hat
        return self.a2

    def backward(self, X, y):
        n = X.shape[0]
        y = y.reshape(-1, 1)

        # output layer error (BCE + sigmoid combine into this simple form)
        dz2 = (self.a2 - y)                      # (n, n_out)
        dW2 = self.a1.T @ dz2 / n
        db2 = dz2.mean(axis=0)

        # hidden layer error, propagated back through W2 and the sigmoid derivative
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.a1 * (1 - self.a1)       # chain rule through sigmoid
        dW1 = X.T @ dz1 / n
        db1 = dz1.mean(axis=0)

        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=5000):
        losses = []
        for _ in range(epochs):
            y_hat = self.forward(X)
            self.backward(X, y)
            losses.append(binary_cross_entropy(y, y_hat.ravel()).mean())
        return losses

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int).ravel()


X_xor_np = XOR[["x1", "x2"]].values.astype(float)
y_xor_np = XOR["y"].values.astype(float)

mlp = SimpleMLP(n_in=2, n_hidden=4, n_out=1, lr=1.0)
losses = mlp.fit(X_xor_np, y_xor_np, epochs=5000)

print("XOR predictions:", mlp.predict(X_xor_np))
print("XOR true labels:", y_xor_np.astype(int))
plt.plot(losses); plt.title("MLP training loss on XOR (BCE)"); plt.xlabel("epoch"); plt.show()

## 8.5 Capstone 2 — revisiting the spiral dataset from Section 1
Section 1 showed a single linear unit fails on the two-spiral dataset. Let's confirm the same
hand-written MLP class (with a wider hidden layer) does much better.

In [ ]:
mlp_spiral = SimpleMLP(n_in=2, n_hidden=16, n_out=1, lr=0.5)
losses_spiral = mlp_spiral.fit(X_spiral, y_spiral.astype(float), epochs=3000)

preds_spiral = mlp_spiral.predict(X_spiral)
print("MLP accuracy on the spiral dataset:", (preds_spiral == y_spiral).mean())
print("(compare to the linear classifier's accuracy computed in Section 1 -- the MLP is dramatically better)")

plt.plot(losses_spiral); plt.title("MLP training loss on the spiral dataset"); plt.xlabel("epoch"); plt.show()

---
# 9. Quick-Reference Summary

## Activation functions

| Function | Formula | Range |
|---|---|---|
| Step | $1$ if $z\ge0$ else $0$ | $\{0,1\}$ |
| Sigmoid | $1/(1+e^{-x})$ | $(0,1)$ |
| Tanh | $(e^x-e^{-x})/(e^x+e^{-x})$ | $(-1,1)$ |
| ReLU | $\max(0,x)$ | $[0,\infty)$ |
| Leaky ReLU | $x$ or $\alpha x$ | $(-\infty,\infty)$ |
| ELU | $x$ or $\alpha(e^x-1)$ | $(-\alpha,\infty)$ |
| Softmax | $e^{x_j}/\sum_k e^{x_k}$ | $(0,1)$, sums to 1 |

## Loss functions

| Loss | Formula | Used for |
|---|---|---|
| MSE | $\frac1n\sum(y-\hat y)^2$ | regression |
| Hinge / perceptron loss | $\frac1n\sum\max(0,-y_if(x_i))$ | linear classifiers, $y\in\{-1,+1\}$ |
| Binary Cross-Entropy | $-[y\log\hat y+(1-y)\log(1-\hat y)]$ | binary classification |
| Categorical Cross-Entropy | $-\sum_c y_c\log\hat y_c$ | multi-class (with softmax) — see Section 10.5 |

## The golden order — concepts, in the order you actually use them
1. Start from a **single artificial neuron** modeled on the biological neuron (Sec. 1).
2. A **perceptron** with a step activation can only draw one straight decision boundary; it learns
   via the discrete perceptron rule, and only converges on **linearly separable** data (Sec. 2–3).
3. Generalize "nudge the weights on error" into **gradient descent**: minimize a smooth **loss
   function** by following its negative gradient (Sec. 4–6).
4. Swap the non-differentiable step function for a **smooth activation** (sigmoid/tanh/ReLU/...) so
   gradient descent actually has a gradient to follow (Sec. 7).
5. **Stack** neurons into layers (an **MLP**) to combine multiple linear boundaries into non-linear
   decision regions — solving problems (XOR, spirals) that a single perceptron cannot (Sec. 8).
6. Train the stack with **backpropagation** — repeated application of the chain rule, layer by layer
   (Sec. 10.1).

---
# 10. Beyond the Modules — Bonus Deep-Dive

*Everything in this section goes beyond what Mod 1–6 covered — added so this notebook is a complete,
self-contained deep-learning foundations reference. Module content stops at Section 9.*

## 10.1 Backpropagation — the chain rule, worked by hand
Backprop is just the **chain rule** applied layer by layer, computing how the loss changes with
respect to *every* weight in the network, starting from the output and working backward (hence
"back"-propagation).

Take the tiny 2-1-1 network used in Section 8.4 ($SimpleMLP$). For a single hidden unit's weight
$w^{[1]}_{11}$ (input 1 -> hidden unit 1), the chain rule says:

$$\frac{\partial L}{\partial w^{[1]}_{11}} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z^{[2]}}\cdot\frac{\partial z^{[2]}}{\partial a^{[1]}_1}\cdot\frac{\partial a^{[1]}_1}{\partial z^{[1]}_1}\cdot\frac{\partial z^{[1]}_1}{\partial w^{[1]}_{11}}$$

Each factor is a *local*, easy derivative:
- $\frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z^{[2]}} = (\hat y - y)$ for BCE + sigmoid combined (a famously clean simplification — this is exactly `dz2` in the code above).
- $\frac{\partial z^{[2]}}{\partial a^{[1]}_1} = w^{[2]}_{1}$ (just the connecting weight).
- $\frac{\partial a^{[1]}_1}{\partial z^{[1]}_1} = a^{[1]}_1(1-a^{[1]}_1)$ (the sigmoid derivative from Sec. 7.1).
- $\frac{\partial z^{[1]}_1}{\partial w^{[1]}_{11}} = x_1$ (the input feeding that weight).

Multiplying these five small, local derivatives together gives the gradient for one weight — and
this is exactly what `dz1 = da1 * self.a1 * (1 - self.a1)` then `dW1 = X.T @ dz1` computed in bulk
(matrix form, over the whole batch) in Section 8.4's `SimpleMLP.backward`.

In [ ]:
# Tiny numeric backprop example: 1 input -> 1 hidden (sigmoid) -> 1 output (sigmoid), by hand
x1 = 0.5
w1_11, b1_1 = 0.8, 0.1     # input -> hidden
w2_1, b2 = -0.6, 0.2       # hidden -> output
y_true = 1.0

# forward
z1 = w1_11 * x1 + b1_1
a1 = sigmoid(z1)
z2 = w2_1 * a1 + b2
y_hat = sigmoid(z2)
loss = binary_cross_entropy(y_true, y_hat)
print(f"forward: z1={z1:.4f} a1={a1:.4f} z2={z2:.4f} y_hat={y_hat:.4f} loss={loss:.4f}")

# backward (chain rule, one scalar at a time)
dL_dyhat_times_dyhat_dz2 = (y_hat - y_true)        # combined BCE+sigmoid gradient
dz2_da1 = w2_1
da1_dz1 = a1 * (1 - a1)
dz1_dw1 = x1

grad_w1_11 = dL_dyhat_times_dyhat_dz2 * dz2_da1 * da1_dz1 * dz1_dw1
print(f"dL/d(w1_11) = {grad_w1_11:.6f}  (this is what gets subtracted, scaled by the learning rate)")

## 10.2 Vanishing & Exploding Gradients
Because backprop **multiplies** many local derivatives together across layers (Sec. 10.1), two
failure modes emerge in deep networks:

- **Vanishing gradients:** if each layer's derivative is $< 1$ (e.g. sigmoid's derivative maxes out
  at $0.25$), the product shrinks toward 0 as it's multiplied across many layers — early layers get
  almost no gradient signal and stop learning. This is *why* ReLU (derivative exactly $1$ for $x>0$)
  became the default over sigmoid/tanh for deep hidden stacks.
- **Exploding gradients:** if weights are large, the product can instead grow unboundedly, causing
  huge, unstable weight updates (often seen as loss suddenly becoming `NaN`).

**Mitigations:** better activations (ReLU family), careful weight initialization (10.3), gradient
clipping (cap the gradient's norm before the update), batch normalization (10.4), residual/skip
connections (architecture-level fix, beyond this notebook's scope).

## 10.3 Weight Initialization
Initializing **all weights to zero** is a classic beginner mistake: every neuron in a layer computes
the exact same output and receives the exact same gradient, so they update identically forever —
the network can never break this symmetry and effectively behaves as a single neuron per layer.

Common fixes initialize weights **randomly**, scaled to keep the variance of activations roughly
stable across layers:
- **Xavier / Glorot init** (good for sigmoid/tanh): $w \sim \mathcal{N}\big(0, \frac{1}{n_{in}}\big)$
- **He init** (good for ReLU): $w \sim \mathcal{N}\big(0, \frac{2}{n_{in}}\big)$

In [ ]:
def xavier_init(n_in, n_out, rng):
    return rng.standard_normal((n_in, n_out)) * np.sqrt(1.0 / n_in)

def he_init(n_in, n_out, rng):
    return rng.standard_normal((n_in, n_out)) * np.sqrt(2.0 / n_in)

rng = np.random.default_rng(0)
W_xavier = xavier_init(256, 256, rng)
W_he = he_init(256, 256, rng)
print("Xavier-init weight std:", W_xavier.std().round(4), " (target ~", round((1/256)**0.5, 4), ")")
print("He-init weight std    :", W_he.std().round(4), " (target ~", round((2/256)**0.5, 4), ")")

## 10.4 Optimizers Beyond Plain Gradient Descent
Plain gradient descent (Sec. 5) takes a fixed-size step opposite the gradient every time. Modern
optimizers improve on this:

- **Momentum:** accumulate a running average ("velocity") of past gradients so the optimizer keeps
  moving through small bumps/noise and speeds up in a consistent direction:
  $$v \leftarrow \beta v + (1-\beta)\nabla L \qquad w \leftarrow w - \eta v$$
- **RMSprop:** divides the learning rate by a running average of *recent squared gradients*, so
  parameters with large/noisy gradients get smaller effective steps and vice versa:
  $$s \leftarrow \beta s + (1-\beta)(\nabla L)^2 \qquad w \leftarrow w - \frac{\eta}{\sqrt{s}+\epsilon}\nabla L$$
- **Adam** (the default in most modern training loops): combines momentum *and* RMSprop's adaptive
  per-parameter scaling.

A quick 1D demo shows momentum "riding through" a shallow, noisy loss surface faster than plain GD.

In [ ]:
def f(w):            # toy loss: a valley with a small bump
    return 0.1 * w**2 + 2 * np.sin(w)

def grad_f(w):
    return 0.2 * w + 2 * np.cos(w)

def run_gd(w0, lr, steps):
    w, path = w0, [w0]
    for _ in range(steps):
        w -= lr * grad_f(w)
        path.append(w)
    return path

def run_momentum(w0, lr, beta, steps):
    w, v, path = w0, 0.0, [w0]
    for _ in range(steps):
        v = beta * v + (1 - beta) * grad_f(w)
        w -= lr * v
        path.append(w)
    return path

path_gd = run_gd(8.0, lr=0.3, steps=40)
path_mom = run_momentum(8.0, lr=0.3, beta=0.9, steps=40)

ws = np.linspace(-10, 10, 200)
plt.plot(ws, f(ws), color="lightgray", label="loss surface")
plt.plot(path_gd, [f(w) for w in path_gd], "o-", label="plain GD", markersize=3)
plt.plot(path_mom, [f(w) for w in path_mom], "o-", label="GD + momentum", markersize=3)
plt.legend(); plt.title("Plain GD vs. Momentum on a bumpy 1D loss")
plt.show()

## 10.5 Regularization
Techniques to stop a network from **overfitting** (memorizing training data instead of generalizing):

- **L2 regularization (weight decay):** add $\frac{\lambda}{2}\sum w^2$ to the loss, penalizing large
  weights and encouraging smaller, more diffuse ones.
- **L1 regularization:** add $\lambda\sum|w|$, which tends to push many weights to *exactly* zero
  (sparsity / implicit feature selection).
- **Dropout:** randomly zero out a fraction $p$ of neurons' outputs on each training step, forcing the
  network to not rely on any single neuron — a cheap approximation of training an ensemble.
- **Early stopping:** stop training once validation loss stops improving, before the model starts
  memorizing training noise.
- **Batch Normalization:** normalize each layer's inputs (zero mean, unit variance) per mini-batch,
  which stabilizes and speeds up training and mildly regularizes as a side effect.

## 10.6 Softmax + Categorical Cross-Entropy — the multi-class generalization of Sections 6.3 & 7.6
For $K$ classes with one-hot true label $y \in \{0,1\}^K$ and softmax output $\hat y$ (Sec. 7.6):
$$L = -\sum_{c=1}^{K} y_c \log(\hat y_c)$$
Since $y$ is one-hot (all zeros except a $1$ at the true class index $t$), every term vanishes except
one, so this collapses to $L = -\log(\hat y_t)$ — exactly the binary case's $-\log(\hat y)$
(Sec. 6.3), generalized from 2 classes to $K$.

In [ ]:
def categorical_cross_entropy(y_true_onehot, y_pred_probs, eps=1e-12):
    y_pred_probs = np.clip(y_pred_probs, eps, 1)
    return -np.sum(y_true_onehot * np.log(y_pred_probs))

# reuse the cat/dog/horse softmax example from Section 7.6
y_true_onehot = np.array([1, 0, 0])     # true class: cat
print("Categorical cross-entropy loss:", categorical_cross_entropy(y_true_onehot, probs))
print("Sanity check, -log(P(cat)):    ", -np.log(probs[0]))

---
## That's the whole map
Biological neuron -> perceptron -> why it fails (XOR) -> gradient descent -> differentiable
activations -> stacking into an MLP -> backprop, better optimizers, and regularization to make it all
train reliably. Everything above is runnable top-to-bottom with just `numpy`, `pandas`, and
`matplotlib`.